# Phân tích dữ liệu MIND và tạo vector biểu diễn

Notebook này chuẩn bị dữ liệu tin tức, tạo vector cho từng tin và tổng hợp thành vector của người dùng.

## 1. Thiết lập môi trường

Import các thư viện cần dùng và cấu hình cách hiển thị số.

Import primary library


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display
import json

np.set_printoptions(
    precision=4,  # 4 chữ số thập phân
    suppress=True,  # không dùng dạng 1.23e-05
    linewidth=200,  # tránh xuống dòng quá sớm
)

### Công cụ xử lý dữ liệu

Nạp các thư viện để tải dữ liệu, thao tác tệp và tạo đặc trưng từ văn bản.

In [ ]:
import os
import kagglehub
import shutil
from sklearn.feature_extraction.text import CountVectorizer

## 2. Tải và chuẩn bị dữ liệu

Tải bộ MIND, lưu các tệp cần thiết vào thư mục làm việc và tạo mẫu dữ liệu nhỏ.

Load Data


In [ ]:
path = kagglehub.dataset_download("arashnic/mind-news-dataset")

print("Dataset downloaded to: ", path)

Copy data to working dir


In [ ]:
source = os.path.join(path, "MINDsmall_train")

destination = r"D:\CDNC\MIND-research\data\raw"

os.makedirs(destination, exist_ok=True)

files = [
    "news.tsv",
    "behaviors.tsv",
    "entity_embedding.vec",
    "relation_embedding.vec",
]

for file in files:
    shutil.copy2(os.path.join(source, file), os.path.join(destination, file))

print("Done")

## 3. Chuẩn bị lịch sử đọc và tin tức

Chọn một nhóm người dùng mẫu, lấy các tin họ đã đọc và lọc thông tin tin tức tương ứng.

**Prepare users behavious dataset**


In [ ]:
behaviours_path = os.path.join(destination, "behaviors.tsv")

columns_behaviours = ["user_id", "time", "history", "impressions"]

behaviours = pd.read_csv(
    behaviours_path,
    sep="\t",
    names=columns_behaviours,
)

behaviours = behaviours[["user_id", "history"]]
behaviours = behaviours.dropna(subset=["history"])

# sample_behaviours = behaviours.sample(n=10, random_state=42).reset_index(drop=True)

output_dir = r"D:\CDNC\MIND-research\data\sample"

behaviours.to_csv(os.path.join(output_dir, "behaviours.csv"), index=False)

print(behaviours.head())

**Get all readed news in user behaviours dataset**


In [ ]:
user_behaviours = behaviours.set_index("user_id")["history"].to_dict()
all_readed_news = set()

for key, value in user_behaviours.items():
    [all_readed_news.add(i) for i in value.split()]

print(len(all_readed_news))

**Get news dataset**


In [ ]:
news_path = os.path.join(destination, "news.tsv")

if not os.path.exists(news_path):
    f_news_small = open(news_path, "x", encoding="utf-8")


columns = [
    "News_ID",
    "Category",
    "SubCategory",
    "Title",
    "Abstract",
    "URL",
    "Title_Entities",
    "Abstract_Entities",
]

news = pd.read_csv(
    news_path,
    sep="\t",
    names=columns,
)

news = news[["News_ID", "Category", "Title"]]

# sample_news = news[news["News_ID"].isin(all_readed_news)].reset_index(drop=True)

output_dir = r"D:\CDNC\MIND-research\data\sample"

news.to_csv(os.path.join(output_dir, "news.csv"), index=False)

print(news.shape)
print(news.head())

## 4. Khởi tạo mô hình

Khởi tạo mô hình embedding câu, mô hình chủ đề và nơi lưu các kết quả vector.

In [ ]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic


class Models:
    def __init__(self):
        self.sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

        self.svectorizer = CountVectorizer(
            stop_words="english",
        )

        self.topic_model = BERTopic(
            calculate_probabilities=True,  # Important to set this to True for probability calculations
            verbose=True,
            vectorizer_model=self.svectorizer,
        )


models = Models()


class VectorContext:
    def __init__(self):
        self.title_list = None
        self.semantic_vector_list = None
        self.probabilities_list = None
        self.topics_vector_list = None

## 5. Tạo vector biểu diễn cho tin tức

Mỗi tin được biểu diễn theo ngữ nghĩa của tiêu đề và phân bố chủ đề.

### Vector Factory

`VectorFactory` được sử dụng để tạo đối tượng biểu diễn vector tương ứng với từng mô hình thông qua một giao diện thống nhất. Thay vì khởi tạo trực tiếp từng lớp, người dùng chỉ cần chỉ định loại mô hình (`sentence` hoặc `bertopic`), Factory sẽ trả về đối tượng phù hợp.

Thiết kế này giúp:

- Tách biệt logic khởi tạo khỏi logic xử lý.
- Dễ dàng thay thế hoặc mở rộng sang các mô hình mới mà không ảnh hưởng đến mã nguồn hiện có.
- Tăng khả năng bảo trì và tái sử dụng mã nguồn.


Vector Interface


In [ ]:
from abc import ABC, abstractmethod


class Vector(ABC):
    @abstractmethod
    def get_vector(self):
        pass

    @abstractmethod
    def overview(self):
        pass

    @abstractmethod
    def summary(self):
        pass

Semantic vector


In [ ]:
class SentenceVector(Vector):
    def __init__(self, model):
        self.model = model
        self.semantic_vector = None

    def get_vector(self, title_list):
        self.title_list = title_list

        titles = [news["title"] for news in title_list]

        vectors = self.model.encode(
            titles, show_progress_bar=True, convert_to_numpy=True
        )

        self.semantic_vector = {
            news["news_id"]: vector for news, vector in zip(title_list, vectors)
        }
        return self.semantic_vector

    def overview(self):
        print("=" * 60)
        print("Sentence Embedding Overview")
        print(f"[Sentence is: {list(self.semantic_vector.keys())[0]}]")
        print("=" * 60)

        print(f"Documents : {len(self.semantic_vector)}")
        print(
            f"Dimension : {self.semantic_vector[list(self.semantic_vector.keys())[0]].shape[0]}"
        )
        print(
            f"Shape     : {self.semantic_vector[list(self.semantic_vector.keys())[0]].shape}"
        )

        print("\nFirst vector (first 10 values):")
        print(self.semantic_vector[list(self.semantic_vector.keys())[0]][:10], "...")

    def summary(self, sample_index=0):

        sample = self.title_list[sample_index]

        news_id = sample["news_id"]
        title = sample["title"]

        vector = self.semantic_vector[news_id]

        metrics = pd.DataFrame(
            {
                "Metric": [
                    "Embedding Model",
                    "Number of Documents",
                    "Embedding Dimension",
                    "Output Shape",
                ],
                "Value": [
                    self.model.__class__.__name__,
                    len(self.semantic_vector),
                    vector.shape[0],
                    str(vector.shape),
                ],
            }
        )

        vector_preview = ", ".join(f"{x:.4f}" for x in vector[:10]) + ", ..."

        example = pd.DataFrame(
            {
                "News ID": [news_id],
                "Sample Title": [title],
                "Embedding (first 10 dims)": [f"[{vector_preview}]"],
            }
        )

        return metrics, example

Topic vector


In [ ]:
class BERTopicVector(Vector):
    def __init__(self, model):
        self.model = model

        self.notice = (
            "BERTopic depends on the sentence transformer model."
            " Please ensure that the sentence transformer model is trained before using BERTopic."
        )

        self.probabilities = None
        self.topics_vector = None

    def get_vector(self, title_list, semantic_vector=None):
        self.title_list = title_list

        titles = [news["title"] for news in title_list]

        embeddings = np.array([semantic_vector[news["news_id"]] for news in title_list])

        if semantic_vector is None:
            print(self.notice)
            return

        topics, probabilities = self.model.fit_transform(
            titles,
            embeddings,
        )

        self.topic_vector = {
            news["news_id"]: {"topic": topic, "probability": probability}
            for news, topic, probability in zip(title_list, topics, probabilities)
        }

        return self.topic_vector

    def overview(self):

        print("=" * 60)
        print("BERTopic Overview")
        print("=" * 60)

        print(f"Number of documents : {len(self.title_list)}")
        print(
            f"Number of topics    : {len(set(v['topic'] for v in self.topic_vector.values()) - {-1})}"
        )

        print("\nTopic distribution:")
        print(self.model.get_topic_info()[["Topic", "Count"]])

        print("\nFirst 5 documents:")

        for sample in self.title_list[:5]:

            news_id = sample["news_id"]
            title = sample["title"]

            topic = self.topic_vector[news_id]["topic"]

            print(f"{news_id}")
            print(f"Title : {title}")
            print(f"Topic : {topic}")
            print("-" * 40)

        first_probability = next(iter(self.topic_vector.values()))["probability"]

        print("\nProbability shape:")
        print(first_probability.shape)

        print("\nFirst 5 probability vectors:")

        for sample in self.title_list[:5]:

            news_id = sample["news_id"]

            probability = self.topic_vector[news_id]["probability"]

            preview = ", ".join(f"{p:.4f}" for p in probability[:10])

            print(f"{news_id} -> [{preview}, ...]")

    def summary(self, sample_index=0):

        # =========================
        # Metrics
        # =========================

        topics = [value["topic"] for value in self.topic_vector.values()]

        first_probability = next(iter(self.topic_vector.values()))["probability"]

        metrics_df = pd.DataFrame(
            {
                "Metric": [
                    "Topic Model",
                    "Number of Documents",
                    "Number of Topics",
                    "Number of Outliers",
                    "Probability Shape",
                ],
                "Value": [
                    self.model.__class__.__name__,
                    len(self.title_list),
                    len(set(topics) - {-1}),
                    np.sum(np.array(topics) == -1),
                    str(first_probability.shape),
                ],
            }
        )

        # =========================
        # Topic Information
        # =========================

        topic_df = self.model.get_topic_info()[["Topic", "Count"]].copy()

        keywords = []

        for topic in topic_df["Topic"]:

            if topic == -1:
                keywords.append("Outlier")
            else:
                words = [word for word, _ in self.model.get_topic(topic)[:5]]
                keywords.append(", ".join(words))

        topic_df["Top Keywords"] = keywords

        # =========================
        # Sample
        # =========================

        sample = self.title_list[sample_index]

        news_id = sample["news_id"]
        title = sample["title"]

        topic_info = self.topic_vector[news_id]

        probs = ", ".join(f"{p:.4f}" for p in topic_info["probability"])

        sample_df = pd.DataFrame(
            {
                "News ID": [news_id],
                "Sample Title": [title],
                "Assigned Topic": [topic_info["topic"]],
                "Probability Distribution": [f"[{probs}]"],
            }
        )

        return metrics_df, topic_df, sample_df

Title list


In [ ]:
model_context = VectorContext()
to_dict = lambda news: {"news_id": news["News_ID"], "title": news["Title"]}
model_context.title_list = [to_dict(news) for _, news in news.iterrows()]

print(model_context.title_list)

Semantic vector


In [ ]:
sentence_vector = SentenceVector(models.sentence_model)

model_context.semantic_vector_list = sentence_vector.get_vector(
    model_context.title_list
)
# print(model_context.semantic_vector_list)
sentence_vector.overview()

metric, example = sentence_vector.summary()

print("\nSummary of Sentence Embedding:")
display(metric)
print("\nExample of Sentence Embedding:")
display(example)

### Tạo vector chủ đề

Gán chủ đề cho từng tiêu đề và hiển thị bản tóm tắt kết quả.

In [ ]:
bertopic_vector = BERTopicVector(models.topic_model)

topics_vector = bertopic_vector.get_vector(
    model_context.title_list, model_context.semantic_vector_list
)
model_context.topics_vector_list = topics_vector
bertopic_vector.overview()

metric, topic, example = bertopic_vector.summary()
print("\nSummary of BERTopic:")
display(metric)
print("\nTopic Information:")
display(topic)
print("\nExample of BERTopic:")
display(example)

In [ ]:
prob = topics_vector["N38324"]["probability"]

print(prob.sum())
print(np.count_nonzero(prob))
print(prob.max())

In [ ]:
import os

os.makedirs("models", exist_ok=True)

bertopic_vector.model.save("models/bertopic_model")
print("Done")

In [ ]:
from bertopic import BERTopic

topic_model_clone = BERTopic.load("models/bertopic_model")

Represented vector


In [ ]:
class RepresentedVector(Vector):
    def __init__(self, title_list, sentence_dict, bertopic_dict):
        self.title_list = title_list
        self.sentence_dict = sentence_dict
        self.bertopic_dict = bertopic_dict
        self.represented_vector = {}

    def get_vector(self):
        self.represented_vector = {
            news["news_id"]: {
                "title": news["title"],
                "semantic": self.sentence_dict[news["news_id"]],
                # "topic": self.bertopic_dict[news["news_id"]]["topic"],
                "topic_distribution": self.bertopic_dict[news["news_id"]][
                    "probability"
                ],
            }
            for news in self.title_list
        }

        return self.represented_vector

    def overview(self):
        print("=" * 80)
        print("Represented Vector Overview")
        print("=" * 80)

        print(f"Documents           : {len(self.represented_vector)}")

        first_news_id = next(iter(self.represented_vector))

        sample = self.represented_vector[first_news_id]

        print(f"Semantic Dimension  : {len(sample['semantic'])}")
        print(f"Topic Distribution  : {len(sample['topic_distribution'])}")
        print(f"Stored Fields       : {list(sample.keys())}")

        print("=" * 80)

    def preview_vector(vector, preview_dims=4):
        vector = [round(float(x), 4) for x in vector]

        if len(vector) <= preview_dims * 2:
            return vector

        return vector[:preview_dims] + ["..."] + vector[-preview_dims:]

    def summary(self, sample_index=0, preview_dims=4):

        news = self.title_list[sample_index]
        news_id = news["news_id"]

        represented = self.represented_vector[news_id]

        semantic = represented["semantic"]
        probability = represented["topic_distribution"]

        def preview(vector):
            vector = [round(float(x), 4) for x in vector]

            if len(vector) <= preview_dims * 2:
                return vector

            return vector[:preview_dims] + ["..."] + vector[-preview_dims:]

        semantic_preview = preview(semantic)
        probability_preview = preview(probability)

        summary_df = pd.DataFrame(
            {
                "Field": [
                    "News ID",
                    "Title",
                    # "Assigned Topic",
                    "Semantic Dimension",
                    "Topic Distribution Dimension",
                ],
                "Value": [
                    news_id,
                    represented["title"],
                    # represented["topic"],
                    len(semantic),
                    len(probability),
                ],
            }
        )

        represented_preview = {
            news_id: {
                "title": represented["title"],
                "semantic": semantic_preview,
                # "topic": represented["topic"],
                "topic_distribution": probability_preview,
            }
        }

        return summary_df, represented_preview

### Kết hợp các vector

Ghép embedding ngữ nghĩa và phân bố chủ đề thành một biểu diễn cho mỗi tin - Represented Vector.

In [ ]:
represented_vector = RepresentedVector(
    model_context.title_list,
    model_context.semantic_vector_list,
    model_context.topics_vector_list,
)

# print(model_context.semantic_vector_list)
# print(model_context.semantic_vector_list)
# print(model_context.topics_vector_list)

represented_vector_list = represented_vector.get_vector()
represented_vector.overview()
represented_vector.summary(sample_index=0, preview_dims=4)

# User Representation Vector


## Prepare sample data User Representation


In [ ]:
user_history = behaviours.set_index("user_id")["history"].to_dict()
print("User mapping: \n", user_history)

## Get Represented Vector of 1 user


### Tính vector người dùng

Lấy trung bình vector của các tin trong lịch sử đọc để tạo biểu diễn cho một người dùng.

In [ ]:
class URV:
    def __init__(self, represented_vector, user_behaviours):
        self.represented_vector = represented_vector
        self.user_behaviours = user_behaviours
        self.user_representation_vector = None
        
    def getURV(self, user_id: str):
        user_history_id = self.user_behaviours[user_id].split()
        
        user_history_vector = [
            self.represented_vector[news_id]
            for news_id in user_history_id
        ]

        user_representation_vector = {
                "user_id": user_id,
                "semantic": np.mean(
                    [v["semantic"] for v in user_history_vector],
                    axis=0
                ),
                "topic_distribution": np.mean(
                    [v["topic_distribution"] for v in user_history_vector],
                    axis=0
                )
            }
        
        self.user_representation_vector = user_representation_vector
        
        assert np.allclose(
            np.mean([v["semantic"] for v in user_history_vector], axis=0),
            sum(v["semantic"] for v in user_history_vector) / len(user_history_vector)
        )
        
        return self.user_representation_vector
        
        
urv = URV(represented_vector=represented_vector_list, user_behaviours=user_history)
urv.getURV("U10339")

#   Recommendation Engine

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def cosine(v1, v2):
    return cosine_similarity(
        v1.reshape(1, -1),
        v2.reshape(1, -1)
    )[0][0]
    
sample_user_vector = urv.getURV("U79199")

In [ ]:
display(sample_user_vector["topic_distribution"])

In [ ]:
class RecommendationEngine:
    def __init__(self, represented_vector, alpha=0.5):
        self.represented_vector = represented_vector
        self.alpha = alpha
        self.score = lambda semantic, topic: self.alpha * semantic + (1 - self.alpha) * topic

    def calculate_similarity(self, user_vector, news_vector):
        semantic = cosine(
            user_vector["semantic"],
            news_vector["semantic"]
        )

        topic = cosine(
            user_vector["topic_distribution"],
            news_vector["topic_distribution"]
        )

        return self.score(semantic, topic), semantic, topic

    def recommend(self, user_vector, candidate_news):
        news_vector = [self.represented_vector[i] for i in candidate_news]
        scores = []
        
        for news_id in candidate_news:
            score, semantic_score, topic_score = self.calculate_similarity(
                user_vector,
                self.represented_vector[news_id]
            )

            scores.append((news_id, score, semantic_score, topic_score))
                
        scores.sort(key=lambda x: x[1], reverse=True)
        display(scores)
        
        

sample = RecommendationEngine(represented_vector_list)
sample.recommend(user_vector=sample_user_vector, candidate_news=["N51048", "N64094", "N13907", "N39010", "N37083", "N459"])
